# تطبيق K-Means لليوم الثاني — التعلم غير الخاضع للإشراف
نستخدم خصائص X فقط، من دون `churned_30d`، ثم نفسر المجموعات بعد التدريب.

## 0) تحميل بيانات منافذ تلقائيًا

In [ ]:
import pandas as pd
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
features = ['orders_per_month','avg_basket_sar','days_since_last_order','promo_usage_rate']
X = df[features].copy()
print('تم تحميل بيانات منافذ ✅', df.shape)
display(X.describe().round(2))

## 1) عوّض المفقود وطبّق Scaling

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
prep = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
Z = prep.fit_transform(X)
print('شكل البيانات بعد التجهيز:', Z.shape)

## 2) مقارنة قيم K من 2 إلى 6 تلقائيًا

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = model.fit_predict(Z)
    rows.append({'K':k, 'Silhouette':silhouette_score(Z, labels), 'Inertia':model.inertia_})
display(pd.DataFrame(rows).round(3))

## 3) مثال محلول باستخدام K = 3

In [ ]:
chosen_k = 3
model = KMeans(n_clusters=chosen_k, n_init=10, random_state=42)
df['cluster'] = model.fit_predict(Z)
profile = df.groupby('cluster')[features].mean().round(2)
display(profile)

## تفسير المجموعات تلقائيًا
تحدد الخلية التالية الخاصية الأكثر اختلافًا في كل مجموعة مقارنة بمتوسط جميع العملاء.

In [ ]:
overall = df[features].mean()
relative = profile - overall
rows = []
for cluster in profile.index:
    strongest = relative.loc[cluster].abs().idxmax()
    direction = 'أعلى من المتوسط' if relative.loc[cluster, strongest] > 0 else 'أقل من المتوسط'
    rows.append({'المجموعة': cluster, 'أبرز خاصية': strongest, 'الوصف': direction})
display(pd.DataFrame(rows))

## الخلاصة
شاهدنا مقارنة قيم K، ثم حلًا كاملًا عند K = 3، وجدول خصائص المجموعات، وتفسيرًا تلقائيًا لأبرز ما يميز كل مجموعة.